**Membuat SparkSession**

In [14]:
from pyspark.sql import SparkSession

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


**A. Membaca dan Eksplorasi Awal**

Baca dataset dari HDFS, tampilkan printSchema(), jumlah baris (count()), dan 10 baris pertama (show(10))

In [15]:
# A. Membaca data dari HDFS
df = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True,
    inferSchema=True
)

# A1. Menampilkan struktur skema
print("=== Struktur Skema ===")
df.printSchema()

# A2. Menghitung jumlah baris
print("=== Jumlah Baris ===")
print("Total baris:", df.count())

# A3. Menampilkan 10 baris pertama
print("=== 10 Baris Pertama ===")
df.show(10)

=== Struktur Skema ===
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

=== Jumlah Baris ===
Total baris: 1000
=== 10 Baris Pertama ===
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|O

**Menangani Data Kosong**

Kolom rating memiliki nilai kosong. Tampilkan berapa banyak, lalu gunakan df.na.fill() atau df.na.drop() (pilih salah satu, jelaskan alasannya pada markdown cell) untuk menanganinya

In [16]:
# B1. Menghitung jumlah nilai kosong di kolom rating
from pyspark.sql.functions import col, when, count as spark_count

print("=== Jumlah Nilai Kosong pada Kolom rating ===")
df.select(
    spark_count(when(col("rating").isNull(), "rating")).alias("rating_kosong")
).show()

# B2. Menangani nilai kosong dengan df.na.fill()
df = df.na.fill({"rating": 0})

# Verifikasi sudah tidak ada yang kosong
print("=== Setelah Penanganan ===")
df.select(
    spark_count(when(col("rating").isNull(), "rating")).alias("rating_kosong")
).show()

=== Jumlah Nilai Kosong pada Kolom rating ===
+-------------+
|rating_kosong|
+-------------+
|          204|
+-------------+

=== Setelah Penanganan ===
+-------------+
|rating_kosong|
+-------------+
|            0|
+-------------+



Saya memilih `df.na.fill()` untuk mengisi nilai kosong pada kolom `rating` dengan angka nol (0), bukan `df.na.drop()`.
Alasannya: menghapus baris (`drop`) akan membuang sekitar 20% data transaksi (karena nilai NaN sengaja dibuat 20% pada dataset), sehingga informasi penjualan menjadi hilang. Mengisi dengan 0 jauh lebih aman karena kita tetap mempertahankan seluruh baris data, dan angka 0 bisa diartikan menjadi "belum ada rating".

**C. Transformasi Data**

Tambahkan kolom `total_pendapatan` (`unit_terjual x harga_satuan`), lalu tambahkan kolom `tier_transaksi` yang bernilai `"Besar"` jika `total_pendapatan > 500000`, atau `"Kecil"` jika sebaliknya

In [17]:
# C1. Menambahkan kolom total_pendapatan
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# C2. Menambahkan kolom tier_transaksi berdasarkan kondisi
df = df.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

# Tampilkan hasil transformasi
print("=== Hasil Transformasi ===")
df.select("order_id", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)

=== Hasil Transformasi ===
+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only sh

**Analisis dengan GroupBy**

1. Kategori apa yang memiliki total_pendapatan tertinggi?

In [18]:
# D1. Kategori dengan total_pendapatan tertinggi
from pyspark.sql.functions import col, sum as spark_sum, count, avg

print("=== D1. Kategori dengan Total Pendapatan Tertinggi ===")
df.groupBy("kategori").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")
).orderBy(col("total_pendapatan").desc()).show()

=== D1. Kategori dengan Total Pendapatan Tertinggi ===
+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+



2. Kota mana dengan jumlah transaksi **tier "Besar"** terbanyak?

In [19]:
# D2. Kota dengan jumlah transaksi tier "Besar" terbanyak
print("=== D2. Kota dengan Transaksi Tier 'Besar' Terbanyak ===")
df.filter(col("tier_transaksi") == "Besar") \
  .groupBy("kota") \
  .count() \
  .orderBy(col("count").desc()) \
  .show()

=== D2. Kota dengan Transaksi Tier 'Besar' Terbanyak ===
+----------+-----+
|      kota|count|
+----------+-----+
|      Solo|   92|
|  Magelang|   78|
|   Kebumen|   78|
|Yogyakarta|   75|
| Purworejo|   66|
|  Semarang|   65|
+----------+-----+



Berapa rata-rata `rating` untuk masing-masing `metode_pembayaran` (data kosong sudah ditangani di bagian B)?

In [20]:
# D3. Rata-rata rating per metode_pembayaran
print("=== D3. Rata-rata Rating per Metode Pembayaran ===")
df.groupBy("metode_pembayaran").agg(
    avg("rating").alias("rata_rata_rating")
).orderBy(col("rata_rata_rating").desc()).show()

=== D3. Rata-rata Rating per Metode Pembayaran ===
+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|         E-Wallet|             3.292|
|     Kartu Kredit|3.1910569105691056|
+-----------------+------------------+



**Menyimpan Hasil ke HDFS**

Simpan DataFrame hasil olahan bagian C (lengkap dengan kolom `total_pendapatan` dan `tier_transaksi`) ke HDFS dalam format CSV baru, kemudian verifikasi apakah sudah berhasil.

In [21]:
# E. Menyimpan DataFrame hasil olahan ke HDFS dalam format CSV
df.select("order_id", "tanggal", "kategori", "kota", "unit_terjual", 
          "harga_satuan", "metode_pembayaran", "rating", 
          "total_pendapatan", "tier_transaksi") \
  .write \
  .mode("overwrite") \
  .option("header", True) \
  .csv("hdfs://localhost:9000/user/mahasiswa/tugas4/output_analisis")

print("Data berhasil disimpan ke HDFS.")

Data berhasil disimpan ke HDFS.
